# Bar-aligned sonograms for a fixed-media staff

This notebook turns the tape audio of a piece into sonogram images that fit the bars of the score. It was written for *Energies*, and it works for any piece with a tape staff exported from Sibelius.

**What goes in**

| Input | Purpose |
|---|---|
| Audio file (WAV, AIFF or FLAC) | The tape. Channels 1 and 2 are analysed. A click on channel 3 is ignored. |
| Tempo map | Says where each bar starts in seconds. Use a MIDI file exported from Sibelius, a CSV, or a constant tempo. |
| Score PDF (optional) | Exported from Sibelius. The notebook reads the barline positions of the tape staff, so each system image follows the engraved bar widths. |

**What comes out** (in the `OUT` folder)

- `bar_001.png` and so on: one image per bar, cropped exactly to the bar.
- `manifest.csv`: start, end, duration and width of every bar.
- `system_01_bars_001-006.png` and so on: one image per system, with every bar at its engraved width.
- `systems.csv`: page, position and width of every system.
- `overview.png`: the whole piece with bar numbers and rehearsal letters.
- A copy of the score PDF with the strips placed on the tape staff, for checking.

**How to use it.** Edit the settings in section 1, then choose *Run All*. Each section ends with a check that shows whether the step worked.

## 0. Install
Run this once. `pdfplumber` reads barlines from the PDF, `pymupdf` places the strips on a copy of the PDF, and `mido` reads the MIDI file.

In [ ]:
%pip install --only-binary=:all: PyMuPDF
%pip install -q numpy scipy matplotlib pillow soundfile mido pdfplumber pymupdf

In [ ]:
import bisect, csv, io, sys, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import soundfile as sf
from PIL import Image
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Settings

Only this cell needs editing. Set `MIDI`, `CSV` or `CONSTANT` to choose the tempo map, and leave the other two as `None`.

`Energies` changes tempo (q = 54 at bar 1, q = 146 at bar 28, q = 54 again at bar 50), so it needs the MIDI export or a CSV. A constant tempo would misplace every bar after bar 27.

In [ ]:
# --- files -----------------------------------------------------------------
AUDIO = "TESTDATA/Energies - Tape Part.wav"      # tape (channels 1-2); a click on channel 3 is ignored
MIDI = "TESTDATA/Energies.mid"          # exported from Sibelius: File > Export > MIDI
CSV = None                        # or "bars.csv" with lines "bar,start_seconds" and a final "end,seconds"
CONSTANT = None                   # or (54, "4/4", 130): tempo, metre, number of bars
PDF = "TESTDATA/Phillip Matty_Energies - Full Score.pdf" # score PDF from Sibelius, or None to skip the system images
OUT = "TESTDATA/strips"                    # output folder

# --- score ---------------------------------------------------------------------
FIRST_BAR = 1
OFFSET = 0.0                      # seconds of audio before bar 1 (the tape has no count-in)
EXPECTED_DURATION = 400.0         # 6'40"; used for a sanity check, or None
MARKS = {"A": 7, "B": 15, "C": 24, "D": 50, "E": 55, "F": 65, "G": 86, "H": 98, "I": 110}
CHECK_BAR = 28                    # the page containing this bar is shown at the end

# --- sonogram ------------------------------------------------------------------
CHANNELS = "1,2"
N_FFT = 4096                      # window length in samples (93 ms at 44.1 kHz)
HOP = 256                         # step between windows in samples (5.8 ms)
FMIN, FMAX = 40, 16000            # frequency range in Hz, on a logarithmic axis
FLOOR_DB = 70                     # dB below the file's peak that print as white
TILT = 0                          # dB per octave of high-frequency lift above 1 kHz
GAMMA = 1.0                       # above 1 darkens quiet detail less, below 1 reveals it

# --- images --------------------------------------------------------------------
DPI = 600
HEIGHT_MM = 14                    # use about 7 for the stamped check copy
MM_PER_SECOND = 12                # width of the per-bar images
STYLE = "ink"                     # "ink": black on white; "transparent": black ink with alpha
CMAP = None                       # a matplotlib colour map name overrides STYLE
INSET_PT = 15                     # space for the clef at the start of each system
BARLINES = 0                      # barline width in pixels on system images (0 = none)

## 2. Tempo map

The tempo map is a list of bar boundaries in seconds. Everything else depends on it, so the notebook checks it against the audio length.

- **MIDI.** Sibelius writes tempo and time-signature events into the file. The notebook converts ticks to seconds across every tempo change, then steps through the bars using the time signatures. Gradual changes are exported as small steps and followed as they come.
- **CSV.** You give the start time of every bar. Any tempo behaviour is possible.
- **Constant.** One tempo and one metre throughout.

The notebook trusts this map and never listens to the tape. If the MIDI and the rendered audio disagree, the strips drift out of their bars. The length check catches a gross mismatch, and the plot below shows each bar's duration, so a tempo change should appear as a step.

In [ ]:
def bars_from_midi(path: str) -> list[float]:
    import mido

    mid = mido.MidiFile(path)
    tpb = mid.ticks_per_beat
    tempos = {0: 500000}
    sigs = {0: (4, 4)}
    end_tick = 0
    for track in mid.tracks:
        t = 0
        for msg in track:
            t += msg.time
            if msg.type == "set_tempo":
                tempos[t] = msg.tempo
            elif msg.type == "time_signature":
                sigs[t] = (msg.numerator, msg.denominator)
        end_tick = max(end_tick, t)

    tl = sorted(tempos.items())
    tl_ticks = [t for t, _ in tl]
    cum = [0.0]
    for (t0, tp0), (t1, _) in zip(tl, tl[1:]):
        cum.append(cum[-1] + (t1 - t0) * tp0 / 1e6 / tpb)

    def seconds(tick: int) -> float:
        i = bisect.bisect_right(tl_ticks, tick) - 1
        return cum[i] + (tick - tl[i][0]) * tl[i][1] / 1e6 / tpb

    sl = sorted(sigs.items())
    sl_ticks = [t for t, _ in sl]
    boundaries = [0]
    t = 0
    while end_tick - t > tpb // 4:
        num, den = sl[bisect.bisect_right(sl_ticks, t) - 1][1]
        t += num * 4 * tpb // den
        boundaries.append(t)
    return [seconds(b) for b in boundaries]


def bars_from_csv(path: str, duration: float) -> list[float]:
    starts, end = [], None
    with open(path, newline="") as f:
        for row in csv.reader(f):
            if len(row) < 2 or not row[0].strip() or row[0].strip().startswith("#"):
                continue
            key = row[0].strip().lower()
            try:
                value = float(row[1])
            except ValueError:
                continue  # header line
            if key == "end":
                end = value
            else:
                starts.append((int(float(key)), value))
    starts.sort()
    bounds = [v for _, v in starts]
    bounds.append(end if end is not None else duration)
    return bounds


def bars_constant(tempo: float, meter: str, n_bars: int) -> list[float]:
    num, den = (int(x) for x in meter.split("/"))
    bar = num * (4 / den) * 60.0 / tempo
    return [i * bar for i in range(n_bars + 1)]

In [ ]:
audio_info = sf.info(AUDIO)
audio_duration = audio_info.frames / audio_info.samplerate - OFFSET

if MIDI:
    bounds = bars_from_midi(MIDI)
elif CSV:
    bounds = bars_from_csv(CSV, audio_duration)
elif CONSTANT:
    bounds = bars_constant(CONSTANT[0], CONSTANT[1], CONSTANT[2])
else:
    raise SystemExit("Set one of MIDI, CSV or CONSTANT.")

n_bars = len(bounds) - 1
durations = np.diff(bounds)
print(f"audio: {audio_duration:.2f} s | tempo map: {n_bars} bars, ends at {bounds[-1]:.2f} s")
if abs(bounds[-1] - audio_duration) > 0.5:
    print(f"WARNING: tempo map and audio differ by {bounds[-1] - audio_duration:+.2f} s. Check the export or OFFSET.")
if EXPECTED_DURATION and abs(bounds[-1] - EXPECTED_DURATION) > 0.5:
    print(f"WARNING: expected {EXPECTED_DURATION:.0f} s, tempo map gives {bounds[-1]:.2f} s.")

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.bar(range(FIRST_BAR, FIRST_BAR + n_bars), durations, color="#1F3A5F", width=0.9)
ax.set_xlabel("bar number"); ax.set_ylabel("bar duration (s)")
ax.set_title("Bar durations from the tempo map (steps mark tempo or metre changes)", fontsize=10)
plt.tight_layout(); plt.show()

## 3. The sonogram

One sonogram is computed for the whole file, then cut into bars. This keeps loudness and frequency scaling identical in every bar, so a quiet bar looks quiet.

| Choice | Setting | Effect |
|---|---|---|
| Window | Hann, `N_FFT` samples | A longer window separates close frequencies and blurs timing. 4096 samples gives about 10.8 Hz resolution and 93 ms of smearing at 44.1 kHz. |
| Step | `HOP` samples | A short step gives a smooth image in time. |
| Frequency axis | Logarithmic, `FMIN` to `FMAX` | Equal distances stand for equal musical intervals. Below about 100 Hz the rows are finer than the FFT bins, so the low end is interpolated. |
| Level | dB below the file peak, `FLOOR_DB` range | Dark means loud. Everything more than `FLOOR_DB` below the peak prints white. |
| Channels | Mean of `CHANNELS` | A mono sum. The click is excluded. |

These settings belong in the methods section of the paper. The last cell of the notebook prints them.

In [ ]:
def load_audio(path: str, channels: str) -> tuple[np.ndarray, int]:
    data, sr = sf.read(path, always_2d=True, dtype="float32")
    idx = [int(c) - 1 for c in channels.split(",")]
    if max(idx) >= data.shape[1]:
        sys.exit(f"File has {data.shape[1]} channel(s); --channels {channels} is not available.")
    return data[:, idx].mean(axis=1), sr


def log_sonogram(x, sr, n_fft, hop, fmin, fmax, rows, floor_db, tilt, gamma):
    """Return (v, dt): v is [rows, frames] in 0..1, low frequency first.

    Frames are computed in blocks so a 400 s file needs little memory."""
    pad = n_fft // 2
    xp = np.pad(x.astype(np.float32), (pad, pad + n_fft))
    n_frames = len(x) // hop + 1
    f = np.fft.rfftfreq(n_fft, 1 / sr)
    grid = np.geomspace(fmin, min(fmax, sr / 2), rows).astype(np.float32)
    win = np.hanning(n_fft).astype(np.float32)
    db = np.empty((rows, n_frames), np.float32)
    block = 1024
    for a in range(0, n_frames, block):
        b = min(a + block, n_frames)
        idx = (np.arange(a, b) * hop)[:, None] + np.arange(n_fft)[None, :]
        mag = np.abs(np.fft.rfft(xp[idx] * win, axis=1)).astype(np.float32)
        d = 20 * np.log10(mag + 1e-9)
        db[:, a:b] = interp1d(f, d, axis=1, kind="linear", assume_sorted=True)(grid).T
    db -= db.max()
    if tilt:
        db += (tilt * np.log2(grid / 1000.0))[:, None].astype(np.float32)
        db -= db.max()
    v = np.clip(1.0 + db / floor_db, 0.0, 1.0) ** gamma
    return v.astype(np.float32), hop / sr


def render_bar(v, dt, t0, t1, px_per_s, style, cmap, width=None):
    width = width or max(1, round((t1 - t0) * px_per_s))
    tt = t0 + (np.arange(width) + 0.5) / width * (t1 - t0)
    pos = np.clip(tt / dt, 0, v.shape[1] - 1.001)
    i0 = pos.astype(int)
    w = (pos - i0).astype(np.float32)
    img = v[:, i0] * (1 - w) + v[:, i0 + 1] * w
    img = img[::-1]  # high frequencies at the top
    if cmap:
        from matplotlib import colormaps
        rgb = (colormaps[cmap](img)[..., :3] * 255).astype(np.uint8)
        return Image.fromarray(rgb, "RGB")
    if style == "transparent":
        rgba = np.zeros(img.shape + (4,), np.uint8)
        rgba[..., 3] = (img * 255).astype(np.uint8)
        return Image.fromarray(rgba, "RGBA")
    grey = (255 * (1 - img)).astype(np.uint8)
    return Image.fromarray(np.dstack([grey] * 3), "RGB")

In [ ]:
x, sr = load_audio(AUDIO, CHANNELS)
if OFFSET > 0:
    x = x[int(round(OFFSET * sr)):]
elif OFFSET < 0:
    x = np.pad(x, (int(round(-OFFSET * sr)), 0))

rows = round(HEIGHT_MM / 25.4 * DPI)
px_per_s = MM_PER_SECOND / 25.4 * DPI
v, dt = log_sonogram(x, sr, N_FFT, HOP, FMIN, FMAX, rows, FLOOR_DB, TILT, GAMMA)
print(f"sonogram: {v.shape[0]} frequency rows x {v.shape[1]} time frames")

The overview below is the whole piece, split into rows, with bar numbers along the bottom of each row and rehearsal letters in bold. Look for the tempo change at bar 28 and for any bar where the events sit visibly off the barlines.

In [ ]:
def overview(v, dt, bounds, first_bar, marks, fmin, fmax, every, n_rows, path):
    import matplotlib
    import matplotlib.pyplot as plt

    total = bounds[-1]
    fig, axes = plt.subplots(n_rows, 1, figsize=(11.7, 8.3), squeeze=False)
    chunk = total / n_rows
    for r, ax in enumerate(axes[:, 0]):
        a, b = r * chunk, (r + 1) * chunk
        ia, ib = int(a / dt), min(int(b / dt) + 1, v.shape[1])
        ax.imshow(1 - v[::-1, ia:ib], aspect="auto", cmap="gray", vmin=0, vmax=1,
                  extent=[ia * dt, ib * dt, np.log10(fmin), np.log10(fmax)])
        ax.set_xlim(a, b)
        ticks = [(bounds[i], first_bar + i) for i in range(len(bounds) - 1)
                 if a - 1e-6 <= bounds[i] < b and (first_bar + i) % every == 0]
        ax.set_xticks([t for t, _ in ticks])
        ax.set_xticklabels([str(n) for _, n in ticks], fontsize=7)
        ax.set_yticks([np.log10(f) for f in (100, 1000, 10000) if fmin <= f <= fmax])
        ax.set_yticklabels([f"{f // 1000}k" if f >= 1000 else str(f)
                            for f in (100, 1000, 10000) if fmin <= f <= fmax], fontsize=7)
        for i in range(len(bounds) - 1):
            if a <= bounds[i] < b:
                ax.axvline(bounds[i], color="#1F3A5F", lw=0.3, alpha=0.5)
        for label, bar in marks.items():
            i = bar - first_bar
            if 0 <= i < len(bounds) - 1 and a <= bounds[i] < b:
                ax.axvline(bounds[i], color="#1F3A5F", lw=1.2)
                ax.text(bounds[i], np.log10(fmax), label, color="#1F3A5F", fontsize=9,
                        fontweight="bold", va="bottom", ha="left")
    axes[-1, 0].set_xlabel("bar number", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()

In [ ]:
Path(OUT).mkdir(parents=True, exist_ok=True)
overview(v, dt, bounds, FIRST_BAR, MARKS, FMIN, FMAX, 5, 4, Path(OUT) / "overview.png")

## 4. One image per bar

Each image is cropped exactly to its bar, and its width is proportional to the bar's duration at `MM_PER_SECOND`. These are the simplest files to use in a paper figure, and they suit a few excerpts.

`manifest.csv` records the start, end and width of every bar.

In [ ]:
out = Path(OUT)
images, manifest = [], []
for i in range(n_bars):
    t0, t1 = bounds[i], bounds[i + 1]
    img = render_bar(v, dt, t0, t1, px_per_s, STYLE, CMAP)
    number = FIRST_BAR + i
    name = f"bar_{number:03d}.png"
    img.save(out / name, dpi=(DPI, DPI))
    images.append(img)
    manifest.append([number, f"{t0:.3f}", f"{t1:.3f}", f"{t1 - t0:.3f}", name,
                     f"{img.width / DPI * 25.4:.1f}"])
with open(out / "manifest.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["bar", "start_s", "end_s", "duration_s", "file", "width_mm"])
    w.writerows(manifest)
print(f"wrote {n_bars} bar images to {out}/")

# preview: bars 1-4 side by side
strip = images[:4]
sheet = Image.new("RGB", (sum(i.width for i in strip) + 20 * len(strip), strip[0].height), "white")
x0 = 0
for im in strip:
    sheet.paste(im.convert("RGBA"), (x0, 0), im.convert("RGBA")); x0 += im.width + 20
sheet.thumbnail((1400, 400)); display(sheet)

## 5. System images that follow the engraved score

A bar's width on the page depends on the note spacing and on how many bars share its system, so the same bar is wider in a three-bar system than in a six-bar one. To match the page, the notebook reads the score PDF:

1. It finds the word "Tape" at the left of each system and the tape staff line beside it.
2. It collects the short vertical barline segments on that staff. Their x positions, in PDF points, are the engraved bar boundaries.
3. It assigns bars to systems in order and draws each bar's sonogram at its engraved width.

The count of bars found on the tape staff must equal the count in the tempo map. A difference means the MIDI export or the PDF is out of step, so the notebook prints a warning.

**Layout caution.** Bar widths belong to the current layout. Lock the format in Sibelius before exporting the PDF, and regenerate the images if the system breaks or spacing change.

In [ ]:
def systems_from_pdf(path: str, label: str = "Tape"):
    """Find the tape staff on every system and the x position of each barline.

    Returns a list of dicts: page (0-based), y, x0, x1 (PDF points) and
    edges (bar boundaries in points, left to right, first = system start).
    Assumes the tape staff is a single line labelled "Tape" at the left of
    each system, which is how Sibelius exports it."""
    import pdfplumber

    systems = []
    with pdfplumber.open(path) as pdf:
        for pi, page in enumerate(pdf.pages):
            hl = [l for l in page.lines
                  if abs(l["top"] - l["bottom"]) < 0.5 and (l["x1"] - l["x0"]) > 250]
            vl = [l for l in page.lines
                  if abs(l["x0"] - l["x1"]) < 0.5 and 4 < (l["bottom"] - l["top"]) < 12]
            for w in page.extract_words():
                if w["text"] != label or w["x0"] > 140:
                    continue
                cy = (w["top"] + w["bottom"]) / 2
                cands = [l for l in hl if abs(l["top"] - cy) < 6]
                if not cands:
                    continue
                line = min(cands, key=lambda l: abs(l["top"] - cy))
                y = line["top"]
                xs = sorted(l["x0"] for l in vl
                            if l["top"] - 0.5 <= y <= l["bottom"] + 0.5
                            and line["x0"] < l["x0"] <= line["x1"] + 1)
                merged = []
                for x in xs:
                    if not merged or x - merged[-1] > 4:
                        merged.append(x)
                end = merged[-1] if merged and line["x1"] - merged[-1] < 15 else line["x1"]
                inner = [x for x in merged if x < end - 4]
                systems.append({"page": pi, "y": y, "x0": line["x0"], "x1": end,
                                "edges": [line["x0"]] + inner + [end]})
    return systems


def build_pdf_systems(v, dt, bounds, systems, first_bar, args, out):
    """One image per system, each bar drawn at its engraved width."""
    ppp = args.dpi / 72.0  # pixels per PDF point
    n_bars = len(bounds) - 1
    k = 0
    rows_out, previews = [], []
    for s, sysm in enumerate(systems):
        edges = list(sysm["edges"])
        n = len(edges) - 1
        mode = "RGBA" if args.style == "transparent" else "RGB"
        bg = (0, 0, 0, 0) if args.style == "transparent" else (255, 255, 255)
        total_w = round((edges[-1] - edges[0]) * ppp)
        canvas = Image.new(mode, (total_w, round(args.height_mm / 25.4 * args.dpi)), bg)
        for j in range(n):
            if k + j >= n_bars:
                break
            left = edges[j] + (args.inset_pt if j == 0 else 0)
            w_px = max(1, round((edges[j + 1] - left) * ppp))
            img = render_bar(v, dt, bounds[k + j], bounds[k + j + 1], 0, args.style, args.cmap, width=w_px)
            canvas.paste(img.convert(mode), (round((left - edges[0]) * ppp), 0))
            if args.barlines:
                for dx in range(args.barlines):
                    for y in range(canvas.height):
                        canvas.putpixel((min(round((edges[j] - edges[0]) * ppp) + dx, total_w - 1), y),
                                        (0, 0, 0, 255) if mode == "RGBA" else (0, 0, 0))
        a, b = first_bar + k, first_bar + min(k + n, n_bars) - 1
        name = f"system_{s + 1:02d}_bars_{a:03d}-{b:03d}.png"
        canvas.save(out / name, dpi=(args.dpi, args.dpi))
        rows_out.append([s + 1, sysm["page"] + 1, f"{sysm['y']:.1f}", f"{edges[0]:.1f}",
                         f"{edges[-1]:.1f}", f"{(edges[-1] - edges[0]) / 72 * 25.4:.1f}", a, b, name])
        previews.append((sysm, out / name))
        k += n
    with open(out / "systems.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["system", "pdf_page", "staff_y_pt", "x_left_pt", "x_right_pt",
                    "width_mm", "first_bar", "last_bar", "file"])
        w.writerows(rows_out)
    return previews


def stamp_pdf(src_pdf, dst_pdf, previews, height_mm):
    """Place each system image over the tape staff, to check alignment on the page."""
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    import fitz

    doc = fitz.open(src_pdf)
    h = height_mm / 25.4 * 72
    for sysm, img in previews:
        page = doc[sysm["page"]]
        rect = fitz.Rect(sysm["x0"], sysm["y"] - h / 2, sysm["x1"], sysm["y"] + h / 2)
        page.insert_image(rect, filename=str(img), overlay=True)
    doc.save(dst_pdf)

In [ ]:
previews, systems = [], []
if PDF:
    systems = systems_from_pdf(PDF)
    pdf_bars = sum(len(s["edges"]) - 1 for s in systems)
    print(f"score PDF: {len(systems)} systems, {pdf_bars} bars on the tape staff")
    if pdf_bars != n_bars:
        print(f"WARNING: the PDF has {pdf_bars} bars and the tempo map has {n_bars}.")
    cfg = SimpleNamespace(dpi=DPI, style=STYLE, height_mm=HEIGHT_MM, cmap=CMAP,
                          inset_pt=INSET_PT, barlines=BARLINES)
    previews = build_pdf_systems(v, dt, bounds, systems, FIRST_BAR, cfg, out)
    print(f"wrote {len(previews)} system images to {out}/")

def bar_range(path):
    a, b = path.stem.split("_bars_")[1].split("-")
    return int(a), int(b)

The first system of the piece and the system containing the tempo change are shown below. Each strip should sit between the barlines of its system.

In [ ]:
shown = [previews[0]] if previews else []
shown += [p for p in previews if bar_range(p[1])[0] <= CHECK_BAR <= bar_range(p[1])[1] and p not in shown]
for sysm, path in shown:
    im = Image.open(path)
    im.thumbnail((1500, 300))
    print(path.name)
    display(im)

## 6. Check against the score

The last check places each system image on the tape staff of a copy of the PDF. Open it and look for three things:

- every strip starts and ends on a barline,
- sound events line up with the cue text above the staff, such as "Pressure Cuts Out" or "Oscillation Enters",
- the final strip ends where the tape ends.

The copy uses the transparent style so the staff and cue text stay visible. A strip taller than the gap between staves overlaps its neighbours, so use a `HEIGHT_MM` of about 7 for this check.

In [ ]:
if PDF and previews:
    import fitz
    stamped = out / "score_with_strips.pdf"
    stamp_pdf(PDF, str(stamped), previews, HEIGHT_MM)
    doc = fitz.open(str(stamped))
    page = next(sysm["page"] for sysm, path in previews
                if bar_range(path)[0] <= CHECK_BAR <= bar_range(path)[1])
    pix = doc[page].get_pixmap(dpi=100)
    display(Image.open(io.BytesIO(pix.tobytes("png"))))
    print(f"wrote {stamped}")

## 7. Record of the method

This cell prints the settings in a form that can go into a paper. It lists what is needed to reproduce the strips.

In [ ]:
print(f"""Sonograms were computed from the mean of channels {CHANNELS} of the {sr} Hz tape file,
using a Hann window of {N_FFT} samples ({N_FFT / sr * 1000:.0f} ms) and a step of {HOP} samples ({HOP / sr * 1000:.1f} ms).
Magnitudes are shown in decibels relative to the peak of the whole file, on a range of {FLOOR_DB} dB,
with a logarithmic frequency axis from {FMIN} Hz to {FMAX / 1000:g} kHz. Dark areas are loud.
Bar boundaries come from the {"MIDI tempo map exported from Sibelius" if MIDI else "supplied tempo map"}
({n_bars} bars, {bounds[-1]:.1f} s). Images are cropped to each bar and set at {DPI} dpi.""")

## Using the images in Sibelius

- Place each system image once, with its left edge on the left edge of the system. `systems.csv` lists the page, the staff position and the width in mm.
- Increase the space around the tape staff so the strip has room. Adding vertical space does not change horizontal bar widths.
- If the layout changes, export a new PDF and run the notebook again.
- The per-bar images suit figures in the paper, where a few excerpts are enough.